## Teacher Training Baseline

* **Objective**: Establish the initial performance baseline.

* **Description**: Standard training pipeline for the RT-DETR v2 model on the SIXray dataset. Focused on basic hyperparameter tuning and model convergence using default configuration settings. Used as a baseline to be improved.

* **Features**: Initial training, standard horizontal flip augmentation, base loss optimization, LR scheduler.

In [1]:
import subprocess
subprocess.run(["git", "clone", "-b", "angelo",
               "https://github.com/angelo4o4/sixray-kd.git",
                "/kaggle/working/sixray-kd"])

import sys
sys.path.append('/kaggle/working/sixray-kd')

!pip install -r /kaggle/working/sixray-kd/requirements.txt --quiet

Cloning into '/kaggle/working/sixray-kd'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

**Imports,device, config and paths**

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Subset, DataLoader

from src.data.dataset import SixRayDataset, collate_fn
from src.data.transforms import build_train_transforms
from src.data.labels import load_label_maps_from_file
from src.models.teacher import load_teacher
from src.engine.trainer import DetectionTrainer
from src.utils.data_utils import (
    create_train_val_split,
    class_distribution,
    print_pos_neg_balance,
)
from src.utils.logger import WandbLogger


print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.10.0+cu128
Using device: cuda


In [ ]:
# Config
MODEL_NAME     = "PekingU/rtdetr_v2_r50vd"
USER           = "angelo"  # or anna
PLATFORM       = "kaggle"
RUN_NAME       = f"01_rtdetr_teacher_baseline_{USER}"
CHECKPOINT_DIR = "/kaggle/working/checkpoints" if PLATFORM == "kaggle" \
                  else "/content/drive/MyDrive/DatasetAPAI/SIXray_Project/checkpoints"
RESUME_FROM    = f"/kaggle/input/datasets/angeloz404/sixray-teacher-checkpoint/{RUN_NAME}_last"  #best
EPOCHS         = 30 # first 30, not to exceed Kaggle GPU limitations
BATCH_SIZE     = 8 # before was 4
LR             = 1e-4
SEED           = 42
TRAIN_TOTAL    = 10500
VAL_POS        = 300
VAL_NEG        = 1200
USE_AUG        = True
FLIP_P         = 0.5
USE_WANDB      = True
WANDB_PROJECT  = "sixray-rtdetr"
EVAL_THRESHOLD = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Paths
LOCAL_EXTRACT_PATH = "/kaggle/input/datasets/angeloz404/sixray10-object-detection"
TRAIN_IMG_DIR      = os.path.join(LOCAL_EXTRACT_PATH, "train", "images")
TRAIN_JSON         = os.path.join(LOCAL_EXTRACT_PATH, "train.json")
TEST_IMG_DIR       = os.path.join(LOCAL_EXTRACT_PATH, "test", "images")
TEST_JSON          = os.path.join(LOCAL_EXTRACT_PATH, "test.json")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

**Model and processor**

In [5]:
id2label, label2id, num_labels = load_label_maps_from_file(TRAIN_JSON)

id2label = {int(k): v for k, v in id2label.items()}
label2id = {v: int(k) for k, v in id2label.items()}

processor, model = load_teacher(MODEL_NAME, id2label, label2id, device=device, use_data_parallel=False)
print(f"Model loaded with {num_labels} classes: {list(id2label.values())}")

preprocessor_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

The image processor of type `RTDetrImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/172M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie class_embed.0.bias to model.decoder.class_embed.1.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.weight to model.decoder.class_embed.1.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.bias to model.decoder.class_embed.2.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.weight to model.decoder.class_embed.2.weight, but both are present in the checkpoints,

Model loaded with 5 classes: ['gun', 'knife', 'wrench', 'pliers', 'scissors']


**Datasets and loaders**

In [ ]:
# Augmentation (just horizontal flip), possibly add brightness and contrast later
from typing import Any


train_transform = build_train_transforms(flip_p = FLIP_P, enabled=USE_AUG)

train_dataset = SixRayDataset(TRAIN_IMG_DIR, TRAIN_JSON, processor, transform=train_transform)
val_dataset   = SixRayDataset(TRAIN_IMG_DIR, TRAIN_JSON, processor, transform=None)
test_dataset  = SixRayDataset(TEST_IMG_DIR, TEST_JSON, processor, transform=None)

train_indices, val_indices, n_pos, n_neg = create_train_val_split(
    train_dataset,
    val_pos = VAL_POS,
    val_neg = VAL_NEG,
    train_total = TRAIN_TOTAL,
    seed = SEED
)

train_subset = Subset(train_dataset, train_indices)
val_subset   = Subset(val_dataset, val_indices)

print(f"Full train set: {n_pos} positive / {n_neg} negative")
print(f"Train subset: {len(train_indices)} images")
print(f"Val subset:   {len(val_indices)} images")
print_pos_neg_balance("Train", train_indices, train_dataset)
print_pos_neg_balance("Val", val_indices, val_dataset)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

batch = next(iter(train_loader))
print(f"\nBatch image shape: {batch['pixel_values'].shape}")

Full train set: 5753 positive / 67049 negative
Train subset: 10500 images
Val subset:   1500 images
Train: 51.9% positives (5453) | 48.1% negatives (5047)
Val: 20.0% positives (300) | 80.0% negatives (1200)

Batch image shape: torch.Size([8, 3, 640, 640])


In [7]:
# Inspecting the classes distribution
train_dist = class_distribution(train_indices, train_dataset, id2label)
val_dist   = class_distribution(val_indices, val_dataset, id2label)

print("Class distribution - Train:")
for cls, count in sorted(train_dist.items()):
    print(f"  {cls}: {count}")

print("Class distribution - Val:")
for cls, count in sorted(val_dist.items()):
    print(f"  {cls}: {count}")

print("\nWarnings:")
for cls in train_dist:
    if val_dist.get(cls, 0) < 10:
        print(f"  {cls} has only {val_dist.get(cls, 0)} val examples")

Class distribution - Train:
  gun: 3290
  knife: 1522
  pliers: 4157
  scissors: 925
  wrench: 2209
Class distribution - Val:
  gun: 175
  knife: 79
  pliers: 225
  scissors: 46
  wrench: 131

Warnings:


**Training** and **Eval**

In [8]:
# WANDB - add checkpoint save with artifacts -
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_key
os.environ["WANDB_SILENT"] = "true"
WANDB_RUN_ID = "sixray_rtdetr_run_01"

logger = WandbLogger(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    id=WANDB_RUN_ID,
    resume="allow",
    config={
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "train_size": len(train_indices),
        "val_size": len(val_indices),
        "use_aug": USE_AUG,
        "flip_p": FLIP_P,
    },
    enabled=USE_WANDB,
)

In [9]:
trainer = DetectionTrainer(
    model=model,
    processor=processor,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    run_name=RUN_NAME,
    lr=LR,
    eval_score_threshold=EVAL_THRESHOLD,
    logger=logger,
    id2label=id2label,
    use_amp=True
)

history = trainer.fit(train_loader, val_loader, epochs=EPOCHS, resume_from=None)

Starting training for 30 epochs (from epoch 1)
Total steps: 39390 | Warmup steps: 3939


Epoch 1/30 (Training):   0%|          | 0/1313 [00:00<?, ?it/s]

End of epoch 1 - Average Loss: 131.2517
  Val mAP: 0.1855 | mAP@50: 0.2726 | mAP@75: 0.1922
    gun: mAP = 0.6406
    knife: mAP = 0.1106
    wrench: mAP = 0.0564
    pliers: mAP = 0.1166
    scissors: mAP = 0.0030


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 30/30 (Training):   0%|          | 0/1313 [00:00<?, ?it/s]

End of epoch 30 - Average Loss: 7.5911
  Val mAP: 0.6264 | mAP@50: 0.7132 | mAP@75: 0.6611
    gun: mAP = 0.9263
    knife: mAP = 0.8097
    wrench: mAP = 0.6938
    pliers: mAP = 0.6531
    scissors: mAP = 0.0491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished!
Best val mAP: 0.6666 at epoch 13


In [10]:
print(f"Val indices hash: {hash(tuple(val_indices))}")

Val indices hash: 6039922416008737244


Cells to download the weights (checkpoint)

In [11]:
import shutil
import os

# Zip dei checkpoint
shutil.make_archive(
    "/kaggle/working/checkpoints_backup",
    "zip",
    "/kaggle/working/checkpoints"
)
print("Done:", os.path.getsize("/kaggle/working/checkpoints_backup.zip") / 1e6, "MB")

Done: 1247.434307 MB


In [12]:
from IPython.display import Javascript
Javascript(f"window.open('/kaggle/working/checkpoints_backup.zip')")

<IPython.core.display.Javascript object>